In [2]:
import google.generativeai as genai
import requests
import re
import json

class ProoomptEngineer:
    key = 'REDACTED_GOOGLE_API_KEY'
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-pro:generateContent?key={key}"
    headers = { 'Content-Type': 'application/json' }

    def _get_llm_response(self, system_prooompt: str, prompt: str) -> str:

        data = {
            "system_instruction": {
                "parts": [
                    {"text": system_prooompt}
                ]
            },
            "contents": {
                "parts": [
                    {"text": prompt}
                ]
            }
        }

        try:
            response = requests.post(self.url, headers=self.headers, data=json.dumps(data))
            response.raise_for_status()
            response_json = response.json()
            text = response_json["candidates"][0]["content"]["parts"][0]["text"]
            return text
        except requests.exceptions.RequestException as e:
            return ""

    def _non_Discord_Code_Block_ificator(self, json_response: str) -> str:
        json_response = re.sub("```$", "", json_response)
        json_response = re.sub("^```json", "", json_response)

        return json_response

    def try_get_name(self, speechToText: str) -> tuple:
        namesystemprompt = """
            Your goal is to take a user input and extract a name if they told you.
            If you didnt find a name you should set name to null.
            If there are mutliple names, set the name to null
            Only reply with raw json in the format of:
            {
                "response": {
                    "name": "extracted name"
                }
            }
        """

        response = self._get_llm_response(namesystemprompt, speechToText)

        if len(response.strip()) == 0:
            return False, None

        response = self._non_Discord_Code_Block_ificator(response)

        name = json.loads(response)['response']['name']

        return (True, name)

# Example Usage
if __name__ == "__main__":
    instruction_file = 'src/systemprompt.txt'
    system_prooompt = ''
    try:
        with open(instruction_file, 'r') as file:
            system_prooompt = file.read().strip()
    except FileNotFoundError:
        system_prooompt = "Provide helpful and concise responses."

    rokosBasilisk = ProoomptEngineer()

    # rokosBasilisk.set_system_prompt(rokosBasilisk, system_prooompt)
    response = rokosBasilisk._get_llm_response(system_prooompt, "hello")

    print(response)


Hello to you too! How can I help you today?



#RAG Tests
